# DeepTuneIO Analyzer — Contention Analysis

Visualización DXT **Offset × Tiempo + histogramas marginales** y **Proceso × Proceso**, reutilizando los pares A/B/C de `Access_Pattern`. Coincidencia no implica degradación medida.

In [26]:
from pathlib import Path
import sys
from IPython.display import display, Image
ROOT = Path.cwd()
ACCESS_PATTERN_MODULE = ROOT.parent / "Access_Pattern" / "Module"
sys.path.insert(0, str(ACCESS_PATTERN_MODULE))
sys.path.insert(0, str(ROOT / "Module"))
from contention_heatmaps import load_pairs, heatmap_with_marginals, process_matrix
from Access_Pattern_Visualization.loader import load_event_csvs


## Configuración

Usar **los mismos CSV de eventos y en el mismo orden** que se pasaron al detector de Access_Pattern. Los `event_a` y `event_b` del CSV de pares son índices posicionales, no identificadores globales.

In [ ]:
PAIRS_CSV = Path(r"CAMBIAR_RUTA/10_contention_three_cases_pairs.csv")
EVENT_CSVS = [Path(r"CAMBIAR_RUTA/eventos_DXT.csv")]
OUTPUT_DIR = ROOT / "results" / "Contention_Analysis"
JOBID = None  # p. ej. 5333197
CASE = "AB"  # AB, A_exact, B_partial o C_shared_resource
PROCESS = None  # p. ej. 28; None = todos
FILE = None  # p. ej. "output_bw_512.hdf5"; None = un mapa por archivo
TIME_BINS = 50
OFFSET_BINS = 40


In [ ]:
if not PAIRS_CSV.is_file():
    raise FileNotFoundError(f"Configura PAIRS_CSV: {PAIRS_CSV}")
missing = [str(p) for p in EVENT_CSVS if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Configura EVENT_CSVS: {missing}")
pairs = load_pairs(PAIRS_CSV)
events = load_event_csvs(EVENT_CSVS)
print("Pares:",len(pairs),"| Eventos DXT cargados:",len(events))
print(pairs.case.value_counts().to_string())


In [ ]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
if CASE != "C_shared_resource":
    figures = heatmap_with_marginals(pairs, events, OUTPUT_DIR,case=CASE,process=PROCESS,file=FILE,time_bins=TIME_BINS,offset_bins=OFFSET_BINS,jobid=JOBID)
else:
    figures=[]
process_fig = process_matrix(pairs,OUTPUT_DIR,case=CASE,process=PROCESS,jobid=JOBID)
if process_fig: figures.append(process_fig)
print("Figuras generadas:",len(figures))
for f in figures:
    print(f.name)
    display(Image(filename=str(f)))
